# ColBERT-style Late Interaction — MaxSim from scratch

Single-vector dense retrieval collapses each document to one embedding. **Late interaction** keeps one embedding *per token* and scores a (query, doc) pair via **MaxSim**:

$$\text{score}(Q, D) = \sum_{q \in Q} \max_{d \in D} \text{sim}(q, d)$$

Each query token picks its *best* doc token. The sum rewards documents that have **at least one strong match per query term** — exactly what entity-heavy queries need.

This notebook implements MaxSim in ~10 lines over the shared hash embedder so you can read the algorithm end-to-end. Production ColBERT uses BERT token embeddings + PLAID compression.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


In [ ]:
import re
import numpy as np
from shared.embedders import cosine_topk, hash_embed
from shared.loaders import load_corpus, load_golden_qa

DOCS = load_corpus()
QA = [q for q in load_golden_qa() if q.source_ids]
doc_texts = [d.title + '. ' + d.abstract for d in DOCS]
doc_ids = [d.arxiv_id for d in DOCS]
DIMS = 128
TOKEN_RE = re.compile(r'[A-Za-z0-9]+')

def tokens(t, cap):
    return [w.lower() for w in TOKEN_RE.findall(t)][:cap]

print(f'{len(DOCS)} docs, {len(QA)} questions')

## Build two indexes side by side

* **Single-vector** — one row per document (`hash_embed(doc_text)`).
* **Late-interaction** — one row per token, organised by document.

In [ ]:
doc_single = hash_embed(doc_texts, dims=DIMS, seed=0)
doc_tokens = [hash_embed(tokens(t, 96) or [''], dims=DIMS, seed=0) for t in doc_texts]
n_token_vecs = sum(m.shape[0] for m in doc_tokens)
print(f'single-vector index: {doc_single.shape[0]} vectors')
print(f'late-interaction index: {n_token_vecs} vectors  ({n_token_vecs/len(DOCS):.1f}x larger)')

## MaxSim scoring

`q_mat @ d_mat.T` gives the full similarity matrix; `np.max(..., axis=1).sum()` collapses it into one score per (Q, D).

In [ ]:
def maxsim(q_mat, d_mat):
    if q_mat.size == 0 or d_mat.size == 0:
        return 0.0
    return float(np.max(q_mat @ d_mat.T, axis=1).sum())

def retrieve_single(q, k=3):
    qv = hash_embed([q], dims=DIMS, seed=0)[0]
    idx, _ = cosine_topk(qv, doc_single, k=k)
    return [doc_ids[i] for i in idx]

def retrieve_maxsim(q, k=3):
    q_mat = hash_embed(tokens(q, 24) or [''], dims=DIMS, seed=0)
    scores = np.array([maxsim(q_mat, d) for d in doc_tokens])
    return [doc_ids[i] for i in np.argsort(-scores)[:k]]

demo = QA[0]
print(f'Q: {demo.question}')
print(f'gold: {demo.source_ids}')
print(f'single  : {retrieve_single(demo.question)}')
print(f'maxsim  : {retrieve_maxsim(demo.question)}')

## recall@{1, 3, 5} comparison

In [ ]:
def recall(retriever, k):
    hits = sum(1 for item in QA if set(retriever(item.question, k)) & set(item.source_ids))
    return round(hits / len(QA), 4)

print(f'{"k":>3}  {"single":>8}  {"maxsim":>8}')
for k in (1, 3, 5):
    print(f'{k:>3}  {recall(retrieve_single, k):>8.4f}  {recall(retrieve_maxsim, k):>8.4f}')

## What you can extend

* Swap `hash_embed` for a real token-level encoder (any HuggingFace BERT) to see the gain on harder queries.
* Add **PLAID-style centroid pruning**: cluster the token vectors with k-means and only score documents whose tokens hit the query's nearest centroids.
* Try **product quantization** on the token vectors (see `02-ivf-pq-quantization/`) to cut the 6–10× index-size penalty.
* For production, use `RAGatouille` or the official `colbert-ai` library.